In [ ]:
%pip install faiss-cpu rapidfuzz sentence_transformers

In [ ]:
%pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [3]:
import pandas as pd
import numpy as np
import os
import kagglehub
import pickle
import json
import requests
import faiss
import torch
from pathlib import Path
from bs4 import BeautifulSoup
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer

Loading the Kaggle Spotify Million Song Dataset

In [4]:
path = kagglehub.dataset_download("notshrirang/spotify-million-song-dataset")
spotify_df = pd.read_csv(f"{path}/spotify_millsongdata.csv")

print(f"Loaded {len(spotify_df)} tracks from Spotify dataset.")
display(spotify_df.head(3))

Loaded 57650 tracks from Spotify dataset.


,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...


Loading the Last.fm tags data

In [5]:
json_files = list(Path("../data/raw").rglob("*.json"))
tag_records = []

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        record = json.load(f)
        tags = record.get("tags", [])
        
        if tags:
            tag_records.append({
                "artist": record.get("artist"),
                "title": record.get("title"),
                "tags": [tag[0] for tag in tags if isinstance(tag, list)]
            })

lastfm_tags_df = pd.DataFrame(tag_records)
print(f"Extracted tags for {len(lastfm_tags_df)} tracks.")

Extracted tags for 1010432 tracks.


Exporting the tags data for faster loading after restarting the kernel

In [6]:
lastfm_tags_df.to_pickle("../data/processed/lastfm_extracted_tags.pkl")
print("Tags successfully exported!")

Tags successfully exported!


In [7]:
tags_file = "../data/processed/lastfm_extracted_tags.pkl"

lastfm_tags_df = pd.read_pickle(tags_file)
print(f"Loaded {len(lastfm_tags_df)} tags.")
display(lastfm_tags_df.head(3))

Loaded 1010432 tags.


,artist,title,tags
0,MRK1,Dr Rudeboy,"[dubstep, UK, Grime, bass, prda, bass floor]"
1,Radio Slave,I Don't Need A Cure For This,"[tech house, elektronischer-b0lzen, haeusliche..."
2,Ibrahim Ferrer,Qué Bueno Baila Usted,"[latin, world, latin jazz, tops, salsa, cuban,..."


Loading the Last.fm lookup CSV and metadata (which contains the genres, moods, and descriptors for the tracks)

In [8]:
lastfm_lookup_df = pd.read_csv("../data/processed/track_metadata_lookup.csv")

print(f"Loaded {len(lastfm_lookup_df)} tracks from local Last.fm metadata.")
display(lastfm_lookup_df.head(3))

Loaded 237662 tracks from local Last.fm metadata.


,track_id,split,title,artist_name,release,year
0,TRAABRX12903CC4816,test,What Child Is This? (Album Version),Plus One,Christmas,2002
1,TRAADFO128F92E1E91,test,In The House,Asher D,Still Kickin',0
2,TRAADQW128F427CE68,test,Prophecy,Xymox,Headclouds,1993


Data cleaning and merging to prepare for fuzzy matching

In [9]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    clean = text.lower().replace(r'[^\w\s]', '').strip()
    clean = clean.replace('feat.', '').replace('ft.', '').strip()
    return clean

spotify_df['clean_artist'] = spotify_df['artist'].apply(clean_text)
spotify_df['clean_title'] = spotify_df['song'].apply(clean_text)

lastfm_lookup_df['clean_artist'] = lastfm_lookup_df['artist_name'].apply(clean_text)
lastfm_lookup_df['clean_title'] = lastfm_lookup_df['title'].apply(clean_text)

In [10]:
lastfm_tags_df['clean_artist'] = lastfm_tags_df['artist'].apply(clean_text)
lastfm_tags_df['clean_title'] = lastfm_tags_df['title'].apply(clean_text)

merged_df = pd.merge(
    spotify_df, 
    lastfm_tags_df[['clean_artist', 'clean_title', 'tags']], 
    on=['clean_artist', 'clean_title'], 
    how='inner'
)

merged_df = merged_df.drop_duplicates(subset=['clean_artist', 'clean_title']).reset_index(drop=True)

print(f"Exactly matched {len(merged_df)} tracks out of {len(spotify_df)} Spotify tracks.")
display(merged_df[['artist', 'song', 'tags']].head())

Exactly matched 11218 tracks out of 57650 Spotify tracks.


,artist,song,tags
0,ABBA,Bang-A-Boomerang,"[pop, 70s, Disco, swedish, Uplifting, fun, cut..."
1,ABBA,Chiquitita,"[pop, 70s, abba, Disco, swedish, oldies, dance..."
2,ABBA,Dancing Queen,"[pop, Disco, 70s, abba, dance, swedish, dancin..."
3,ABBA,Disillusion,"[pop, abba, 70s, 80s, swedish, agnetha, 1970's..."
4,ABBA,Does Your Mother Know,"[pop, 70s, abba, Disco, swedish, dance, rock, ..."


Loading BGE-M3 to use Nvidia CUDA and GPU VRAM

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-m3", device=device)

print(f"Model loaded on: {model.device}")
print(f"Max Sequence Length: {model.max_seq_length}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded on: cuda:0
Max Sequence Length: 8192


Fuzzy matching unmatched Spotify tracks

In [12]:
unmatched_spotify = spotify_df.merge(merged_df[['clean_artist', 'clean_title']], on=['clean_artist', 'clean_title'], how='left', indicator=True)
unmatched_spotify = unmatched_spotify[unmatched_spotify['_merge'] == 'left_only'].drop(columns=['_merge'])

unique_unmatched_artists = unmatched_spotify['clean_artist'].unique()
lastfm_unique_artists = lastfm_tags_df['clean_artist'].unique().tolist()

print(f"Fuzzy matching {len(unique_unmatched_artists)} unique artists...")
artist_mapping = {}

for artist in unique_unmatched_artists:
    match = process.extractOne(artist, lastfm_unique_artists, scorer=fuzz.token_sort_ratio, score_cutoff=80)
    if match:
        artist_mapping[artist] = match[0]

print(f"Found fuzzy matches for {len(artist_mapping)} artists.")
print("Matching titles within those artist groups...")

fuzzy_matches = []
for spot_artist, group in unmatched_spotify.groupby('clean_artist'):
    if spot_artist not in artist_mapping:
        continue
        
    lfm_artist = artist_mapping[spot_artist]
    lastfm_artist_tracks = lastfm_tags_df[lastfm_tags_df['clean_artist'] == lfm_artist]
    lastfm_titles = lastfm_artist_tracks['clean_title'].tolist()
    
    for _, row in group.iterrows():
        title_match = process.extractOne(row['clean_title'], lastfm_titles, scorer=fuzz.token_sort_ratio, score_cutoff=80)
        
        if title_match:
            matched_title = title_match[0]
            tags = lastfm_artist_tracks[lastfm_artist_tracks['clean_title'] == matched_title]['tags'].values[0]
            
            row_dict = row.to_dict()
            row_dict['tags'] = tags
            fuzzy_matches.append(row_dict)

fuzzy_df = pd.DataFrame(fuzzy_matches)
final_df = pd.concat([merged_df, fuzzy_df], ignore_index=True)

final_df = final_df.drop_duplicates(subset=['clean_artist', 'clean_title']).reset_index(drop=True)

print(f"Recovered {len(fuzzy_df)} tracks. Total dataset size: {len(final_df)}")
display(final_df.head(3))

Fuzzy matching 643 unique artists...
Found fuzzy matches for 572 artists.
Matching titles within those artist groups...
Recovered 1410 tracks. Total dataset size: 12628


,artist,song,link,text,clean_artist,clean_title,tags
0,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...,abba,bang-a-boomerang,"[pop, 70s, Disco, swedish, Uplifting, fun, cut..."
1,ABBA,Chiquitita,/a/abba/chiquitita_20002978.html,"Chiquitita, tell me what's wrong \r\nYou're e...",abba,chiquitita,"[pop, 70s, abba, Disco, swedish, oldies, dance..."
2,ABBA,Dancing Queen,/a/abba/dancing+queen_20002554.html,"You can dance, you can jive, having the time o...",abba,dancing queen,"[pop, Disco, 70s, abba, dance, swedish, dancin..."


Using genre data from Every Noise at Once to filter out genres from within the track tags

In [13]:
print("Fetching genres from Every Noise at Once...")
response = requests.get("https://everynoise.com/")
soup = BeautifulSoup(response.text, "html.parser")

every_noise_genres = {
    div.text.strip().lower().replace('»', '').strip() 
    for div in soup.find_all('div', class_='genre')
}
print(f"Loaded {len(every_noise_genres)} unique genres.")

normalized_noise_genres = {
    g.replace('-', '').replace(' ', '').replace('_', '').replace('/', '').replace("'", ""): g 
    for g in every_noise_genres
}

custom_genres = {
    'heavymetal': 'heavy metal',
    'punkrock': 'punk rock',
    'hardrock': 'hard rock',
    'britishpunk': 'british punk',
    'rocknroll': 'rock and roll',
    'rnb': 'r&b'
}
normalized_noise_genres.update(custom_genres)

def split_genres_normalized(tag_list):
    if not isinstance(tag_list, list):
        return [], []
    
    genres, descriptors = [], []
    
    for tag in tag_list:
        lower_tag = tag.lower()
        norm_tag = lower_tag.replace('-', '').replace(' ', '').replace('_', '').replace('/', '').replace("'", "")
        
        if norm_tag in normalized_noise_genres:
            genres.append(normalized_noise_genres[norm_tag]) 
        else:
            descriptors.append(lower_tag)
            
    return list(set(genres)), descriptors

final_df['genres'], final_df['tags'] = zip(*final_df['tags'].map(split_genres_normalized))
final_df.to_pickle("../data/processed/final_merged_data.pkl")

display(final_df[['artist', 'song', 'genres', 'tags']].head())

Fetching genres from Every Noise at Once...
Loaded 6291 unique genres.


,artist,song,genres,tags
0,ABBA,Bang-A-Boomerang,"[rock, boston punk, easy listening, novelty, e...","[70s, swedish, uplifting, fun, cute, abba-esqu..."
1,ABBA,Chiquitita,"[soft rock, rock, easy listening, pop rock, eu...","[70s, abba, swedish, oldies, dance, female voc..."
2,ABBA,Dancing Queen,"[soft rock, rock, classic rock, swedish pop, s...","[70s, abba, dance, swedish, dancing queen, old..."
3,ABBA,Disillusion,"[disco, pop, singer-songwriter]","[abba, 70s, 80s, swedish, agnetha, 1970's, bro..."
4,ABBA,Does Your Mother Know,"[rock, classic rock, swedish pop, pop rock, eu...","[70s, abba, swedish, dance, 1979, 80s, bad str..."


In [14]:
# final_df.to_csv(r'final_df.csv')

Encoding and exporting the FAISS index (L2 distance)

In [15]:
lyrics_list = final_df['text'].tolist()

print("Generating embeddings...")
embeddings = model.encode(lyrics_list, batch_size=32, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(index, "../data/processed/lyric_embeddings.index")
final_df.to_pickle("../data/processed/final_merged_data.pkl")

print(f"Successfully indexed {index.ntotal} tracks.")

Generating embeddings...


Batches:   0%|          | 0/395 [00:00<?, ?it/s]

Successfully indexed 12628 tracks.
